# Grasp Intensity Model Training

This notebook fine-tunes a Vision Transformer (ViT) to predict hand grasp intensity (0-1) using the FreiHAND dataset.

In [ ]:
from google.colab import drive
import os
import sys

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set working directory
working_dir = '/content/drive/MyDrive/Colab Notebooks/CARMA'
os.chdir(working_dir)
print(f"Current Directory: {os.getcwd()}")

# Add scripts to path for imports
sys.path.append(os.path.join(working_dir, 'scripts'))

In [ ]:
# 3. Install missing libraries
!pip install -q transformers ultralytics cv2_enumerate_cameras

## Unzip Dataset to Local Runtime
Unzipping to the local Colab disk (`/content/`) is significantly faster for training than reading directly from Google Drive.

In [ ]:
# Path to your zip file on Drive (update if necessary)
zip_path = 'data/FreiHAND_pub_v2.zip'
local_data_path = '/content/freihand'

if os.path.exists(zip_path) and not os.path.exists(local_data_path):
    print(f"Unzipping {zip_path} to {local_data_path}...")
    !unzip -q "{zip_path}" -d "{local_data_path}"
    print("Unzip complete.")
else:
    print("Data already unzipped.")

## Data Preparation
Calculate the grasp intensity labels from the raw FreiHAND 3D keypoints.

In [ ]:
from grasp_intensity.data_prep import prepare_labels

# prepare_labels takes the ROOT of the FreiHAND dataset (where training_xyz.json is)
output_label_path = 'data/grasp_intensity/grasp_labels.csv'
os.makedirs('data/grasp_intensity', exist_ok=True)

prepare_labels(local_data_path, output_label_path)

## Sanity Check Visualization
Visualize images across the 0.0 to 1.0 intensity range.

In [ ]:
from grasp_intensity.visualize_labels import visualize_bins

%matplotlib inline
visualize_bins(csv_path=output_label_path, img_dir=os.path.join(local_data_path, 'training/rgb'))

## Model Training
Fine-tune the Vision Transformer using the local fast storage.

In [ ]:
from grasp_intensity.train_vit import train

# 1. Save checkpoints to Drive so we can resume if Colab fails
checkpoint_path = 'models/grasp_intensity_checkpoints'
os.makedirs(checkpoint_path, exist_ok=True)

# 2. Use local images for speed, but keep labels and models on Drive
train(
    img_dir=os.path.join(local_data_path, 'training/rgb'),
    csv_path=output_label_path,
    output_dir='models/grasp_intensity_best',
    checkpoint_dir=checkpoint_path
)